In [15]:
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os
import dagshub

In [3]:
dagshub.init(repo_owner='AmanGupta8423', repo_name='mlops-mini-project', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow')

mlflow.set_experiment("Bow vs Tfidf")

Accessing as AmanGupta8423

Initialized MLflow to track repo "AmanGupta8423/mlops-mini-project"

Repository AmanGupta8423/mlops-mini-project initialized!

2026/11/15 14:27:04 INFO mlflow.tracking.fluent: Experiment with name 'Bow vs Tfidf' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/50e70f2cf67846a6b1808ee05ab13f52', creation_time=1794733025775, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1794733025775, lifecycle_stage='active', name='Bow vs Tfidf', tags={}, trace_location=None, workspace='default'>

In [4]:
def lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

In [5]:
def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

In [19]:
def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

In [7]:
def lower_case(text):
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

In [8]:
def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub("\s+", ' ', text).strip()
    return text

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
C:\Users\win11\AppData\Local\Temp\ipykernel_11424\3673323208.py:5: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [9]:
def removing_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

In [11]:
def normalize_text(df):
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise


In [20]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [21]:
df = normalize_text(df)

x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


In [22]:
vectorizer = {
    'BoW':CountVectorizer(),
    'TF-IDF':TfidfVectorizer()
}

algorithms = {
    'LogisticRegression':LogisticRegression(),
    'MultinomialNB':MultinomialNB(),
    'XGBoost':XGBClassifier(),
    'RandomForest':RandomForestClassifier(),
    'GradientBoosting':GradientBoostingClassifier()
}

In [32]:
with mlflow.start_run(run_name="All Experiments") as parent_run:
    for algo_name, algorithm in algorithms.items():
        for vec_name, vec in vectorizer.items():
            with mlflow.start_run(run_name=f'{algo_name} with {vec_name}',nested=True) as child_run:
                X = vec.fit_transform(df['content'])
                y = df['sentiment'].astype('int64')
                X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state = 42)


                mlflow.log_param('vectorizer',vec_name)
                mlflow.log_param('algorithm',algo_name)
                mlflow.log_param('test_size', 0.2)

                model = algorithm
                model.fit(X_train,y_train)

                if algo_name == 'LogisticRegression':
                    mlflow.log_param('C',model.C)
                elif algo_name == 'MultinomialNB':
                    mlflow.log_param("alpha", model.alpha)
                elif algo_name == 'XGBoost':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                elif algo_name == 'RandomForest':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("max_depth", model.max_depth)
                elif algo_name == 'GradientBoosting':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                    mlflow.log_param("max_depth", model.max_depth)

                y_pred = model.predict(X_test)
                accuracy = accuracy_score(y_test,y_pred)
                precision = precision_score(y_test, y_pred)
                recall = recall_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)


                mlflow.log_metrics({
                    'accuracy':accuracy,
                    'precision':precision,
                    'recall':recall,
                    'f1_score':f1
                })

                mlflow.sklearn.log_model(
                sk_model=model,
                name="model",
                serialization_format="skops",
                skops_trusted_types=[
                    "xgboost.core.Booster",
                    "xgboost.sklearn.XGBClassifier",
                    "sklearn.tree._tree.Tree"
                ])

                mlflow.log_artifact('exp2.ipynb')

                print(f"Algorithm: {algo_name}, Feature Engineering: {vec_name}")
                print(f"Accuracy: {accuracy}")
                print(f"Precision: {precision}")
                print(f"Recall: {recall}")
                print(f"F1 Score: {f1}")

Algorithm: LogisticRegression, Feature Engineering: BoW
Accuracy: 0.7937349397590362
Precision: 0.7846750727449079
Recall: 0.7970443349753694
F1 Score: 0.7908113391984359
🏃 View run LogisticRegression with BoW at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/2/runs/550f5ed0bd454ac9adae4ca8d4effdab
🧪 View experiment at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/2
Algorithm: LogisticRegression, Feature Engineering: TF-IDF
Accuracy: 0.7942168674698795
Precision: 0.777882797731569
Recall: 0.8108374384236453
F1 Score: 0.79401833092137
🏃 View run LogisticRegression with TF-IDF at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/2/runs/31e460c40dc64d6e868612256a8edb71
🧪 View experiment at: https://dagshub.com/AmanGupta8423/mlops-mini-project.mlflow/#/experiments/2
Algorithm: MultinomialNB, Feature Engineering: BoW
Accuracy: 0.7826506024096386
Precision: 0.7797619047619048
Recall: 0.774384236453202
F1 Score:

In [27]:
df.sample(5)

,sentiment,content
6560,1,thanks everyone unfortunately nothing done sal...
32129,1,got back cheescake factory boy oh boy gut stuffed
10252,0,day without sleep migraine thought life postr ...
11179,0,chezery im going work married wud never see
26428,1,quot i love knowledge love thinkin fressshman ...
